# Bone Fracture Detection from X-ray Images
### Deep Learning Assessment — MSc Data Science, Manchester Metropolitan University

**Task:** Binary classification of X-ray images into Fractured / Not Fractured  
**Models:** Custom CNN · VGG16 (Transfer Learning)  
**Dataset:** Bone Fracture Dataset — Kaggle (17,000 images)  
**Key result:** Both models achieve ~99% test accuracy; VGG16 converges in 3 epochs vs 10 for the custom CNN

---
## Notebook Structure
1. Setup & Configuration  
2. Data Loading & Exploration  
3. Data Generators  
4. Custom CNN — Architecture, Training & Evaluation  
5. VGG16 — Transfer Learning, Training & Evaluation  
6. Model Comparison  
7. Grad-CAM Visualisation

## 1. Setup & Configuration

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from tqdm import tqdm

import tensorflow as tf
import keras
from keras.applications import VGG16
from keras.models import Model, Sequential
from keras.layers import (Conv2D, MaxPool2D, BatchNormalization,
                          Dropout, Flatten, Dense)
from keras.preprocessing.image import ImageDataGenerator
from keras.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# ── Global Configuration ──────────────────────────────────────────────────────
# Update DATA_DIR to point to your dataset root folder
DATA_DIR    = '/content/drive/MyDrive/datasetBone'
TRAIN_DIR   = os.path.join(DATA_DIR, 'train')
VAL_DIR     = os.path.join(DATA_DIR, 'val')
TEST_DIR    = os.path.join(DATA_DIR, 'test')

IMAGE_SIZE  = (224, 224)
BATCH_SIZE  = 32
EPOCHS_CNN  = 10
EPOCHS_VGG  = 3
CLASS_NAMES = ['fractured', 'not fractured']

print('Configuration loaded.')

## 2. Data Loading & Exploration

The dataset contains X-ray images organised into  and  
subdirectories for each split. We build a flat DataFrame of image paths and labels 
to feed into Keras data generators.

In [ ]:
def load_data(dataset_path):
    """
    Walk a directory of class-named subdirectories and return a DataFrame
    with columns: image (full path), label (subdirectory name).
    """
    images, labels = [], []
    for subfolder in sorted(os.listdir(dataset_path)):
        subfolder_path = os.path.join(dataset_path, subfolder)
        if not os.path.isdir(subfolder_path):
            continue
        for fname in os.listdir(subfolder_path):
            if fname.lower().endswith('.jpg'):
                images.append(os.path.join(subfolder_path, fname))
                labels.append(subfolder)
    return pd.DataFrame({'image': images, 'label': labels})

train_df = load_data(TRAIN_DIR)
val_df   = load_data(VAL_DIR)
test_df  = load_data(TEST_DIR)

print(f'Train: {len(train_df)} images')
print(f'Val:   {len(val_df)} images')
print(f'Test:  {len(test_df)} images')
print(f'
Class distribution (train):
{train_df["label"].value_counts()}')

In [ ]:
# ── Class Distribution Plot ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colours = ['#E07B54', '#5B8DB8', '#6BAF6B']

for ax, (df, title, colour) in zip(axes, [
    (train_df, 'Training Set',   colours[0]),
    (val_df,   'Validation Set', colours[1]),
    (test_df,  'Test Set',       colours[2]),
]):
    counts = df['label'].value_counts()
    ax.bar(counts.index, counts.values, color=colour, edgecolor='white')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Class'); ax.set_ylabel('Count')
    ax.grid(axis='y', alpha=0.3)
    for i, (cls, val) in enumerate(counts.items()):
        ax.text(i, val + 30, str(val), ha='center', fontsize=11)

plt.suptitle('Class Distribution Across Splits', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Sample Image Grid ────────────────────────────────────────────────────────
def show_sample_grid(df, n=16, title='Sample X-ray Images'):
    """Display a random grid of images with their class labels."""
    indices = np.random.randint(0, len(df), n)
    fig, axes = plt.subplots(4, 4, figsize=(14, 14))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    for ax, idx in zip(axes.flatten(), indices):
        img = cv2.imread(df.image.iloc[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img); ax.axis('off')
        ax.set_title(df.label.iloc[idx].title(), fontsize=10)
    plt.tight_layout()
    plt.show()

show_sample_grid(train_df, title='Sample Training X-rays')

## 3. Data Generators

Keras  handles on-the-fly loading and normalisation.
Images are rescaled to [0, 1] and resized to 224×224 to match ImageNet input conventions.
Binary class mode is used since this is a two-class problem (fractured / not fractured).

In [ ]:
datagen = ImageDataGenerator(rescale=1.0 / 255)

train_generator = datagen.flow_from_dataframe(
    train_df, x_col='image', y_col='label',
    target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=True
)
val_generator = datagen.flow_from_dataframe(
    val_df, x_col='image', y_col='label',
    target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=False
)
test_generator = datagen.flow_from_dataframe(
    test_df, x_col='image', y_col='label',
    target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=False
)

print(f'Class indices: {train_generator.class_indices}')

## 4. Custom CNN

A bespoke CNN built from scratch using Keras Sequential API.

**Architecture:**
- 3 convolutional blocks: Conv2D → BatchNormalization → MaxPool2D → Dropout(0.3)
- Filters: 32 → 64 → 128 (progressively deeper feature extraction)
- Fully connected head: Dense(256) → Dropout → Dense(128) → Dropout → Dense(1, sigmoid)
- Loss: Binary cross-entropy | Optimiser: Adam | Metrics: accuracy, AUC, specificity

**Design rationale:** BatchNorm stabilises training; progressive filter depth captures 
increasingly abstract features; Dropout at 0.3 prevents overfitting on the 13K training set.

In [ ]:
def build_custom_cnn():
    """
    Custom CNN for binary bone fracture classification.
    Three convolutional blocks with progressive filter depth (32→64→128),
    BatchNormalization, MaxPooling, and Dropout regularisation.
    """
    model = Sequential([
        # Block 1
        Conv2D(32, (3, 3), activation='relu', input_shape=(*IMAGE_SIZE, 3)),
        BatchNormalization(),
        MaxPool2D((2, 2)),

        # Block 2
        Conv2D(64, (3, 3), activation='relu'),
        BatchNormalization(),
        MaxPool2D((2, 2)),
        Dropout(0.3),

        # Block 3
        Conv2D(128, (3, 3), activation='relu'),
        BatchNormalization(),
        MaxPool2D((2, 2)),
        Dropout(0.3),

        # Classification head
        Flatten(),
        Dense(256, activation='relu'),
        Dropout(0.3),
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid'),
    ], name='custom_cnn')
    return model

cnn_model = build_custom_cnn()
cnn_model.summary()

In [ ]:
# ── Compile & Train Custom CNN ───────────────────────────────────────────────
cnn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy',
             keras.metrics.SpecificityAtSensitivity(0.5),
             keras.metrics.AUC()]
)

cnn_callbacks = [
    ModelCheckpoint('cnn_best_model.keras', save_best_only=True, monitor='val_loss'),
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
]

print('Training Custom CNN...')
hist_cnn = cnn_model.fit(
    train_generator,
    epochs=EPOCHS_CNN,
    validation_data=val_generator,
    callbacks=cnn_callbacks
)

In [ ]:
# ── Plot CNN Training Curves ──────────────────────────────────────────────────
def plot_training_curves(history, model_name):
    """Plot accuracy, loss, specificity and AUC curves for a training history."""
    h = pd.DataFrame(history.history)
    acc_col  = [c for c in h.columns if c == 'accuracy'][0]
    vacc_col = [c for c in h.columns if c == 'val_accuracy'][0]
    auc_col  = [c for c in h.columns if 'auc' in c and 'val' not in c][0]
    vauc_col = [c for c in h.columns if 'val_auc' in c][0]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'{model_name} — Training Curves', fontsize=13, fontweight='bold')

    axes[0].plot(h[acc_col],  'b-o', label='Train Accuracy')
    axes[0].plot(h[vacc_col], 'r-o', label='Val Accuracy')
    axes[0].plot(h['loss'],     'b--', label='Train Loss', alpha=0.6)
    axes[0].plot(h['val_loss'], 'r--', label='Val Loss',   alpha=0.6)
    axes[0].set_title('Accuracy & Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(h[auc_col],  'b-o', label='Train AUC')
    axes[1].plot(h[vauc_col], 'r-o', label='Val AUC')
    axes[1].set_title('AUC'); axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_training_curves(hist_cnn, 'Custom CNN')

In [ ]:
# ── Evaluate Custom CNN ───────────────────────────────────────────────────────
print('Evaluating Custom CNN on test set...')
cnn_loss, cnn_acc, cnn_spec, cnn_auc = cnn_model.evaluate(test_generator, verbose=1)
print(f'
Test Loss:        {cnn_loss:.4f}')
print(f'Test Accuracy:    {cnn_acc:.4f}')
print(f'Test Specificity: {cnn_spec:.4f}')
print(f'Test AUC:         {cnn_auc:.4f}')

# Predictions
y_true      = test_generator.classes
y_pred_prob = cnn_model.predict(test_generator, verbose=1)
y_pred_cnn  = (y_pred_prob >= 0.5).astype(int).ravel()
y_true      = y_true.ravel()

print('
Classification Report (Custom CNN):')
print(classification_report(y_true, y_pred_cnn, target_names=['Fractured', 'Not Fractured']))

In [ ]:
# ── Confusion Matrix — Custom CNN ─────────────────────────────────────────────
def plot_confusion_matrix(y_true, y_pred, class_names, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(title, fontsize=13, fontweight='bold')
    plt.xlabel('Predicted'); plt.ylabel('True')
    plt.tight_layout(); plt.show()

plot_confusion_matrix(y_true, y_pred_cnn,
                      ['Fractured', 'Not Fractured'],
                      'Confusion Matrix — Custom CNN')

## 5. VGG16 — Transfer Learning

VGG16 pre-trained on ImageNet is used as a frozen feature extractor. 
A custom classification head is added on top.

**Why transfer learning?** ImageNet features (edges, textures, shapes) transfer 
well to medical X-ray images, enabling the model to converge in just 3 epochs 
compared to 10 for the custom CNN — with equivalent accuracy.

**Architecture additions:**
- Flatten → Dense(256, ReLU) → Dropout(0.3) → Dense(128, ReLU) → Dropout(0.3) → Dense(1, sigmoid)
- All VGG16 base layers frozen during training

In [ ]:
def build_vgg16_model():
    """
    VGG16 with frozen ImageNet weights as a feature extractor,
    with a custom binary classification head.
    """
    base = VGG16(weights='imagenet', include_top=False, input_shape=(*IMAGE_SIZE, 3))
    for layer in base.layers:
        layer.trainable = False

    x = Flatten()(base.output)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.3)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.3)(x)
    out = Dense(1, activation='sigmoid')(x)

    return Model(inputs=base.input, outputs=out, name='vgg16_transfer')

vgg16_model = build_vgg16_model()
vgg16_model.summary()

In [ ]:
# ── Compile & Train VGG16 ────────────────────────────────────────────────────
vgg16_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy',
             keras.metrics.SpecificityAtSensitivity(0.5),
             keras.metrics.AUC()]
)

vgg16_callbacks = [
    ModelCheckpoint('vgg16_best_model.keras', save_best_only=True, monitor='val_loss'),
    EarlyStopping(patience=3, restore_best_weights=True, verbose=1),
]

print('Training VGG16...')
hist_vgg16 = vgg16_model.fit(
    train_generator,
    epochs=EPOCHS_VGG,
    validation_data=val_generator,
    callbacks=vgg16_callbacks
)

In [ ]:
# ── Evaluate VGG16 ───────────────────────────────────────────────────────────
plot_training_curves(hist_vgg16, 'VGG16 Transfer Learning')

print('Evaluating VGG16 on test set...')
vgg_loss, vgg_acc, vgg_spec, vgg_auc = vgg16_model.evaluate(test_generator, verbose=1)
print(f'
Test Loss:        {vgg_loss:.4f}')
print(f'Test Accuracy:    {vgg_acc:.4f}')
print(f'Test Specificity: {vgg_spec:.4f}')
print(f'Test AUC:         {vgg_auc:.4f}')

y_pred_prob_vgg = vgg16_model.predict(test_generator, verbose=1)
y_pred_vgg      = (y_pred_prob_vgg >= 0.5).astype(int).ravel()

print('
Classification Report (VGG16):')
print(classification_report(y_true, y_pred_vgg, target_names=['Fractured', 'Not Fractured']))

plot_confusion_matrix(y_true, y_pred_vgg,
                      ['Fractured', 'Not Fractured'],
                      'Confusion Matrix — VGG16')

## 6. Model Comparison

Side-by-side comparison of the custom CNN and VGG16 on classification metrics,
training efficiency, and complexity.

In [ ]:
# ── Comparative Classification Report ────────────────────────────────────────
report_cnn   = classification_report(y_true, y_pred_cnn,
                                     target_names=['Fractured', 'Not Fractured'],
                                     output_dict=True)
report_vgg16 = classification_report(y_true, y_pred_vgg,
                                     target_names=['Fractured', 'Not Fractured'],
                                     output_dict=True)

report_cnn_df   = pd.DataFrame(report_cnn).transpose()
report_vgg16_df = pd.DataFrame(report_vgg16).transpose()

comparison_df = report_cnn_df.join(report_vgg16_df, lsuffix='_CNN', rsuffix='_VGG16')
print('Comparative Classification Report:')
print(comparison_df.round(4))

In [ ]:
# ── Summary Table ─────────────────────────────────────────────────────────────
summary = pd.DataFrame({
    'Model':          ['Custom CNN', 'VGG16'],
    'Test Accuracy':  [f'{cnn_acc:.1%}',  f'{vgg_acc:.1%}'],
    'Test AUC':       [f'{cnn_auc:.4f}',  f'{vgg_auc:.4f}'],
    'Test Specificity':[f'{cnn_spec:.4f}', f'{vgg_spec:.4f}'],
    'Epochs to Train': [EPOCHS_CNN, EPOCHS_VGG],
    'Trainable Params':['~1.2M', '~0.5M (head only)'],
})
print(summary.to_string(index=False))

## 7. Grad-CAM Visualisation

Gradient-weighted Class Activation Mapping (Grad-CAM) highlights the image regions 
most influential in the model's prediction — providing interpretability critical 
for clinical AI applications.

For each X-ray, the heatmap shows which areas of the bone the model is "looking at" 
when making its fracture/no-fracture decision. Clinicians can use this to validate 
that the model is focusing on anatomically meaningful regions rather than image artefacts.

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """
    Generate a Grad-CAM heatmap for a single image.

    Args:
        img_array:           Preprocessed image array (1, H, W, 3)
        model:               Trained Keras model
        last_conv_layer_name: Name of the final convolutional layer
        pred_index:          Class index to explain (None = argmax)

    Returns:
        heatmap: 2D numpy array normalised to [0, 1]
    """
    # Build a model that outputs (last_conv_layer, final_predictions)
    grad_model = Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    # Gradients of the predicted class w.r.t. last conv layer output
    grads  = tape.gradient(class_channel, conv_outputs)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def overlay_gradcam(img_path, heatmap, alpha=0.4):
    """Overlay a Grad-CAM heatmap on the original image."""
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, IMAGE_SIZE)

    heatmap_resized = cv2.resize(heatmap, IMAGE_SIZE)
    heatmap_coloured = np.uint8(255 * heatmap_resized)
    heatmap_coloured = cv2.applyColorMap(heatmap_coloured, cv2.COLORMAP_JET)
    heatmap_coloured = cv2.cvtColor(heatmap_coloured, cv2.COLOR_BGR2RGB)

    overlay = cv2.addWeighted(img, 1 - alpha, heatmap_coloured, alpha, 0)
    return img, overlay


def display_gradcam_grid(df, model, last_conv_layer, n_samples=6, title='Grad-CAM'):
    """Display original and Grad-CAM overlaid images for a sample of test images."""
    samples = df.sample(n_samples, random_state=42)
    fig, axes = plt.subplots(n_samples, 2, figsize=(10, 4 * n_samples))
    fig.suptitle(title, fontsize=14, fontweight='bold')

    for row, (_, sample) in enumerate(samples.iterrows()):
        # Preprocess
        img_arr = tf.keras.utils.load_img(sample['image'], target_size=IMAGE_SIZE)
        img_arr = tf.keras.utils.img_to_array(img_arr) / 255.0
        img_arr = np.expand_dims(img_arr, axis=0)

        heatmap = make_gradcam_heatmap(img_arr, model, last_conv_layer)
        original, overlay = overlay_gradcam(sample['image'], heatmap)

        pred_prob = model.predict(img_arr, verbose=0)[0][0]
        pred_cls  = 'Not Fractured' if pred_prob >= 0.5 else 'Fractured'
        true_cls  = sample['label'].title()

        axes[row, 0].imshow(original)
        axes[row, 0].set_title(f'Original — True: {true_cls}', fontsize=10)
        axes[row, 0].axis('off')

        axes[row, 1].imshow(overlay)
        axes[row, 1].set_title(f'Grad-CAM — Pred: {pred_cls} ({pred_prob:.2f})', fontsize=10)
        axes[row, 1].axis('off')

    plt.tight_layout()
    plt.show()

print('Grad-CAM utilities defined.')

In [ ]:
# ── Grad-CAM: Custom CNN ─────────────────────────────────────────────────────
# The last convolutional layer in our custom CNN is named 'conv2d_2'
# (third Conv2D block). Confirm with: cnn_model.summary()

CNN_LAST_CONV = 'conv2d_2'
display_gradcam_grid(test_df, cnn_model, CNN_LAST_CONV,
                     n_samples=6, title='Grad-CAM — Custom CNN')

In [ ]:
# ── Grad-CAM: VGG16 ──────────────────────────────────────────────────────────
# The last convolutional layer in VGG16 is 'block5_conv3'

VGG_LAST_CONV = 'block5_conv3'
display_gradcam_grid(test_df, vgg16_model, VGG_LAST_CONV,
                     n_samples=6, title='Grad-CAM — VGG16')